In [1]:
!pip install flask flask-cors pyngrok pillow -q

In [3]:
from google.colab import files
uploaded = files.upload()

Saving model.jpeg to model (1).jpeg


In [1]:
!pip install flask flask-cors pyngrok pillow -q

In [ ]:




from pyngrok import ngrok, conf
conf.get_default().auth_token = "YOUR_NGROK_AUTH_TOKEN"  # Replace with your actual ngrok auth token

from flask import Flask, request, jsonify
from flask_cors import CORS
from gradio_client import Client, handle_file
from PIL import Image
import tempfile, os, threading, base64, traceback

app = Flask(__name__)
CORS(app)

hf_client = Client("zhengchong/CatVTON", token="hf_YourHuggingFaceToken")  # Replace with your actual Hugging Face token

@app.route('/tryon', methods=['POST'])
def tryon():
    try:
        person_file = request.files['person']
        garment_file = request.files['garment']

        # Save person as PNG (keep original format, don't force JPEG)
        person_img = Image.open(person_file.stream).convert('RGB')
        with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as p:
            person_path = p.name
            person_img.save(person_path, 'PNG')

        # Save cloth as JPEG
        garment_img = Image.open(garment_file.stream).convert('RGB')
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as g:
            garment_path = g.name
            garment_img.save(garment_path, 'JPEG')

        result = hf_client.predict(
            person_image={
                "background": handle_file(person_path),
                "layers": [],
                "composite": handle_file(person_path),
                "id": None
            },
            cloth_image=handle_file(garment_path),
            num_inference_steps=20,
            guidance_scale=2.5,
            seed=42,
            api_name="/submit_function_p2p"
        )

        os.unlink(person_path)
        os.unlink(garment_path)

        result_path = result["path"] if isinstance(result, dict) else result

        with open(result_path, 'rb') as f:
            result_b64 = base64.b64encode(f.read()).decode()

        return jsonify({
            'success': True,
            'result_image': f'data:image/webp;base64,{result_b64}'
        })

    except Exception as e:
        traceback.print_exc()
        return jsonify({'success': False, 'error': str(e)})

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'running'})

public_url = ngrok.connect(5000)
print(f"\n✅ API URL: {public_url}\n")
print("Paste this URL into your tryon.html as API_URL!")

threading.Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()

Loaded as API: https://zhengchong-catvton.hf.space

✅ API URL: NgrokTunnel: "https://latrine-suffering-legged.ngrok-free.dev" -> "http://localhost:5000"

Paste this URL into your tryon.html as API_URL!


In [3]:
from google.colab import files
uploaded = files.upload()

Saving model.jpeg to model (2).jpeg


In [4]:
import requests

filename = list(uploaded.keys())[0]
files_payload = {
    'person': (filename, open(filename, 'rb'), 'image/jpeg'),
    'garment': (filename, open(filename, 'rb'), 'image/jpeg'),
}
r = requests.post("http://localhost:5000/tryon", files=files_payload)

print(r.status_code)
print(r.text[:500])

INFO:werkzeug:127.0.0.1 - - [05/Jul/2026 11:36:33] "POST /tryon HTTP/1.1" 200 -


200
{"result_image":"
